In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

In [2]:
transaction_pd = pd.read_csv('../../data/raw/transactions_train.csv')
customer_df = pd.read_csv('../../data/raw/customers.csv')
art_df = pd.read_csv('../../data/raw/articles.csv')

In [3]:
print(len(transaction_pd))
print(len(customer_df))
print(len(art_df))

31788324
1371980
105542


In [4]:
art_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 105542 entries, 0 to 105541
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype
---  ------                        --------------   -----
 0   article_id                    105542 non-null  int64
 1   product_code                  105542 non-null  int64
 2   prod_name                     105542 non-null  str  
 3   product_type_no               105542 non-null  int64
 4   product_type_name             105542 non-null  str  
 5   product_group_name            105542 non-null  str  
 6   graphical_appearance_no       105542 non-null  int64
 7   graphical_appearance_name     105542 non-null  str  
 8   colour_group_code             105542 non-null  int64
 9   colour_group_name             105542 non-null  str  
 10  perceived_colour_value_id     105542 non-null  int64
 11  perceived_colour_value_name   105542 non-null  str  
 12  perceived_colour_master_id    105542 non-null  int64
 13  perceived_colour_master_n

In [5]:
# transaction & customer 중복 key 컬럼 확인
print(set(transaction_pd.columns)&set(customer_df.columns))

# customer & transaction merge (key : customer id)
cust_tran_df = customer_df.merge(transaction_pd, how='inner', on = ['customer_id'])

# cust_tran_df.isna().sum()

{'customer_id'}


In [6]:
set(art_df.columns)&set(cust_tran_df.columns)
total_df = cust_tran_df.merge(art_df, how='inner', on =['article_id'])

In [7]:
# 패션 뉴스, 온라인 마케팅 등록 여부
total_df['FN'] = total_df['FN'].fillna(0)
total_df['Active'] = total_df['Active'].fillna(0)

In [8]:
# FN, ACtive 0 채우기 확인
# total_df.isna().sum()

### club_member_status 
- H&M 클럽 멤버십은 회원이 계정을 생성해 가입한 뒤, 구매·리뷰·기타 활동을 통해 포인트를 적립하고 이를 리워드로 교환하는 로열티 프로그램
- ACTIVE: 현재 클럽 멤버십이 활성화된 회원
- PRE-CREATE: 계정/멤버십이 사전 생성되었으나 완전 활성화 전 상태
- LEFT CLUB: 클럽을 탈퇴한 회원 (브랜드 완전이탈이 확정이 아님, 조금 애매함)
- NaN: 멤버십 상태 정보가 없는 결측값

**H&M club_member_status NaN

오프라인 구매만 한 고객,
멤버십 상태 수집 전 가입자,
데이터 병합 과정에서 상태 누락,
멤버십과 무관한 일반 고객

In [9]:
total_df['club_member_status'].unique()

<StringArray>
['ACTIVE', nan, 'PRE-CREATE', 'LEFT CLUB']
Length: 4, dtype: str

In [10]:
# club_member_status na값 삭제
# 62,000(NA) / 30,000,000 ≈ 0.21%
total_df['club_member_status'] = total_df['club_member_status'].replace('nan',np.nan)
total_df = total_df.dropna(subset=['club_member_status'])

In [11]:
# total_df.isna().sum()

#### 패션 뉴스레터 수신 빈도 
- array(['NONE', 'Regularly', nan, 'Monthly'], dtype=object)

| 값         | 의미              |
| --------- | --------------- |
| NONE      | 수신 안 함 /뉴스레터 거부자와 정보 없음 고객을 같은 그룹|
| Monthly   | 월 1회 수신         |
| Regularly | 정기 수신 (월 1회 이상) |
| NaN       | 정보 없음           |


In [12]:
total_df['fashion_news_frequency'] = total_df['fashion_news_frequency'].fillna('UNKNOWN')
print(total_df['fashion_news_frequency'].unique())
total_df.isna().sum()

<StringArray>
['NONE', 'Regularly', 'UNKNOWN', 'Monthly']
Length: 4, dtype: str


customer_id                          0
FN                                   0
Active                               0
club_member_status                   0
fashion_news_frequency               0
age                             126598
postal_code                          0
t_dat                                0
article_id                           0
price                                0
sales_channel_id                     0
product_code                         0
prod_name                            0
product_type_no                      0
product_type_name                    0
product_group_name                   0
graphical_appearance_no              0
graphical_appearance_name            0
colour_group_code                    0
colour_group_name                    0
perceived_colour_value_id            0
perceived_colour_value_name          0
perceived_colour_master_id           0
perceived_colour_master_name         0
department_no                        0
department_name          

### 나이 (결측치 알수없음으로 채움)
- min(10대), max(90대) - 이상치인지 확인 필요
 (16.0, 99.0)


In [13]:
min(total_df['age']), max(total_df['age'])

(16.0, 99.0)

In [14]:
total_df[total_df['age']>=90].value_counts() # 1393건

customer_id                                                       FN   Active  club_member_status  fashion_news_frequency  age   postal_code                                                       t_dat       article_id  price     sales_channel_id  product_code  prod_name                 product_type_no  product_type_name  product_group_name  graphical_appearance_no  graphical_appearance_name  colour_group_code  colour_group_name  perceived_colour_value_id  perceived_colour_value_name  perceived_colour_master_id  perceived_colour_master_name  department_no  department_name       index_code  index_name        index_group_no  index_group_name  section_no  section_name                    garment_group_no  garment_group_name  detail_desc                                                                                                                                                                                                                                     
bc870dbac4a150f6e5cf8e1d71e59

In [15]:
bins = [0, 19, 29, 39, 49, 59, 69, 100]
labels = ['10대 미만', '10대', '20대', '30대', '40대', '50대', '60대 이상']
total_df['age_cut'] = pd.cut(total_df['age'], bins=bins, labels=labels, right=True).astype('str')
type(total_df['age_cut'])
total_df['age_cut'] = total_df['age_cut'].str.replace('nan', 'UNKNOWN')


In [16]:
total_df['age_cut'].unique()

<StringArray>
['30대', '10대', '40대', '20대', '60대 이상', '50대', '10대 미만', nan]
Length: 8, dtype: str

In [17]:
total_df['age_cut'].value_counts()

age_cut
10대       13039351
20대        6416888
40대        5130097
30대        4901185
50대        1201949
10대 미만      690192
60대 이상      219899
Name: count, dtype: int64

In [18]:
total_df.columns

Index(['customer_id', 'FN', 'Active', 'club_member_status',
       'fashion_news_frequency', 'age', 'postal_code', 't_dat', 'article_id',
       'price', 'sales_channel_id', 'product_code', 'prod_name',
       'product_type_no', 'product_type_name', 'product_group_name',
       'graphical_appearance_no', 'graphical_appearance_name',
       'colour_group_code', 'colour_group_name', 'perceived_colour_value_id',
       'perceived_colour_value_name', 'perceived_colour_master_id',
       'perceived_colour_master_name', 'department_no', 'department_name',
       'index_code', 'index_name', 'index_group_no', 'index_group_name',
       'section_no', 'section_name', 'garment_group_no', 'garment_group_name',
       'detail_desc', 'age_cut'],
      dtype='str')

### 1차 컬럼 분류

In [19]:
total_df = total_df[['customer_id', 'FN', 'club_member_status',
       'fashion_news_frequency', 'age', 't_dat', 'price', 'product_group_name','age_cut']]
EDA_columns = total_df

In [20]:
EDA_columns.head(5)

,customer_id,FN,club_member_status,fashion_news_frequency,age,t_dat,price,product_group_name,age_cut
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.044051,Garment Upper body,30대
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.035576,Garment Upper body,30대
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.030492,Garment Upper body,30대
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-02,0.010153,Garment Full body,30대
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-25,0.050831,Garment Upper body,30대


### price 값 수치 의미 (기업 미공개- 정규화된 데이터)

| 값 예시                          | 해석                      |                 |
| -------------------------- | ----------------------- | --------------- |
| `0.009`                    | 원본에서 매우 낮은 가격 → 스케일링 결과 |                 |
| `0.0278` (dataset average) | 스케일링된 평균 거래 가격          | ([ARAMDAUN][1]) |
| `0.5915` (max)             | 스케일된 가장 비싼 상품           | ([ARAMDAUN][1]) |

[1]: https://ars420.tistory.com/42?utm_source=chatgpt.com "[Kaggle] H&M Personalized Fashion Recommendations"


In [21]:
type(EDA_columns['t_dat'])
EDA_columns['transaction_dat'] = pd.to_datetime(EDA_columns['t_dat'], format="%Y-%m-%d")

In [22]:
# 평균 구매 간격
EDA_columns = EDA_columns.sort_values(['customer_id','transaction_dat'])
EDA_columns['prev_date'] = EDA_columns.groupby('customer_id')['transaction_dat'].shift(1)
EDA_columns['gap'] = (EDA_columns['transaction_dat'] - EDA_columns['prev_date']).dt.days
EDA_columns = EDA_columns.fillna(0) # 시작날은 시차 0 


In [23]:
EDA_columns['t_dat'] = EDA_columns['t_dat'].astype(str)

# 1. 전체 고객 이탈 label : API 적용-> API가 90일보다 크면 이탈 (90일) + 1번 구매 후 없음

In [24]:
# API ratio = 고객 구매 주기 일수의 총합 / 거래 횟수 - 1(첫번쨰 거래 제외)

API_count = EDA_columns.groupby('customer_id').agg({
    't_dat': lambda x: x.nunique(),  # 중복 제거한 거래일 수 
    'gap': 'sum'                     # gap 합계
}).reset_index()

API_count['API ratio'] = (API_count['gap']) / (API_count['t_dat']-1)
API_count     

# # API, 신규고객 이탈 라벨링

# API
API_count['churn'] = np.where(API_count['API ratio']> 90 , 1, 0) # 이탈 - 1 / 미이탈 - 0 / 이커머스 기준 90일 (데이터리안)

# 신규고객 
API_count['only_first_tran'] = np.where(API_count['t_dat']==1,1,0) # 0인 고객들은 API 수식이 적용이 안됨

# API + 신규
API_count['total_churn'] = API_count['churn']+API_count['only_first_tran'] # api 비율 기준 + 거래가 1인 고객은 무조건 이탈
API_count

,customer_id,t_dat,gap,API ratio,churn,only_first_tran,total_churn
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,10,618.0,68.666667,0,0,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,23,656.0,29.818182,0,0,0
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,7,726.0,121.000000,1,0,1
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,1,0.0,NaN,0,1,1
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,6,670.0,134.000000,1,0,1
...,...,...,...,...,...,...,...
1356222,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,11,522.0,52.200000,0,0,0
1356223,ffffcd5046a6143d29a04fb8c424ce494a76e5cdf4fab5...,19,693.0,38.500000,0,0,0
1356224,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,19,720.0,40.000000,0,0,0
1356225,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,4,78.0,26.000000,0,0,0


In [25]:
API_count['API ratio'] = API_count['API ratio'].fillna(0)
API_count.isna().sum()
API_count

,customer_id,t_dat,gap,API ratio,churn,only_first_tran,total_churn
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,10,618.0,68.666667,0,0,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,23,656.0,29.818182,0,0,0
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,7,726.0,121.000000,1,0,1
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,1,0.0,0.000000,0,1,1
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,6,670.0,134.000000,1,0,1
...,...,...,...,...,...,...,...
1356222,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,11,522.0,52.200000,0,0,0
1356223,ffffcd5046a6143d29a04fb8c424ce494a76e5cdf4fab5...,19,693.0,38.500000,0,0,0
1356224,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,19,720.0,40.000000,0,0,0
1356225,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,4,78.0,26.000000,0,0,0


# 2. 신규고객 이탈 (첫날 이후 구매가 없는 고객)

In [26]:
# 거래 및 거래가 1번밖에 없는 사람은 신규고객 이탈
purchase_counts = EDA_columns['customer_id'].value_counts()

# 라벨링: 구매 1회 → 1, 그 외 → 0
churn_label = pd.DataFrame({
    'customer_id': purchase_counts.index,
    '신규고객이탈': (purchase_counts == 1).astype(int)  # 이탈 → 1, 미이탈 → 0
})
churn_label = churn_label.reset_index(drop=True)
churn_label

,customer_id,신규고객이탈
0,be1981ab818cf4ef6765b2ecaea7a2cbf14ccd6e8a7ee9...,0
1,b4db5e5259234574edfff958e170fe3a5e13b6f146752c...,0
2,49beaacac0c7801c2ce2d189efe525fe80b5d37e46ed05...,0
3,a65f77281a528bf5c1e9f270141d601d116e1df33bf9df...,0
4,cd04ec2726dd58a8c753e0d6423e57716fd9ebcf2f14ed...,0
...,...,...
1356222,fffdaa06e7f3e9698fb1df460b03ca6cc56528b98982c5...,1
1356223,fffdfad0d0527fa55b97f0d2f2ae4b2e659de8345bfd73...,1
1356224,fffe61b99c2d0418ed22190a8490b142247e8897c67941...,1
1356225,ffffaff3905b803d1c7e153a1378a5151e1f34f236ba54...,1


In [27]:
### 메모리 부족으로 인한 타입변경


# API_count['total_churn'] = API_count['total_churn'].astype(np.int8)
# churn_label['신규고객이탈'] = churn_label['신규고객이탈'].astype(np.int8)
# EDA_columns['customer_id'] = EDA_columns['customer_id'].astype('category')
# API_count['customer_id'] = API_count['customer_id'].astype('category')
# churn_label['customer_id'] = churn_label['customer_id'].astype('category')

In [28]:
# 전체 고객 이탈 라벨링 데이터를 EDA df에 merge (0:미이탈 / 1:이탈)
API_merge = API_count[['customer_id','API ratio','total_churn']]
API_churn_label_merge = EDA_columns.merge(API_merge,on=['customer_id'],how='left')

# 신규 고객 이탈 라벨링 데이터를 EDA df에 merge (0:미이탈 / 1:이탈)
first_customer_merge = churn_label[['customer_id','신규고객이탈']]
API_churn_label_merge = API_churn_label_merge.merge(first_customer_merge,on=['customer_id'],how='left' )
API_churn_label_merge



,customer_id,FN,club_member_status,fashion_news_frequency,age,t_dat,price,product_group_name,age_cut,transaction_dat,prev_date,gap,API ratio,total_churn,신규고객이탈
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.044051,Garment Upper body,30대,2018-12-27,0,0.0,68.666667,0,0
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.035576,Garment Upper body,30대,2018-12-27,2018-12-27 00:00:00,0.0,68.666667,0,0
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.030492,Garment Upper body,30대,2018-12-27,2018-12-27 00:00:00,0.0,68.666667,0,0
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-02,0.010153,Garment Full body,30대,2019-05-02,2018-12-27 00:00:00,126.0,68.666667,0,0
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-25,0.050831,Garment Upper body,30대,2019-05-25,2019-05-02 00:00:00,23.0,68.666667,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31726154,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,1.0,ACTIVE,Regularly,18.0,2020-04-09,0.043203,Garment Full body,10대 미만,2020-04-09,2020-04-05 00:00:00,4.0,26.000000,0,0
31726155,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,1.0,ACTIVE,Regularly,18.0,2020-04-09,0.013542,Garment Upper body,10대 미만,2020-04-09,2020-04-09 00:00:00,0.0,26.000000,0,0
31726156,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,1.0,ACTIVE,Regularly,18.0,2020-04-25,0.050831,Garment Full body,10대 미만,2020-04-25,2020-04-09 00:00:00,16.0,26.000000,0,0
31726157,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,1.0,ACTIVE,Regularly,18.0,2020-06-22,0.016932,Underwear,10대 미만,2020-06-22,2020-04-25 00:00:00,58.0,26.000000,0,0


In [29]:
API_churn_label_merge[API_churn_label_merge['customer_id']=='ffffd9ac14e89946416d80e791d064701994755c3ab686a1eaf3458c36f52241']

,customer_id,FN,club_member_status,fashion_news_frequency,age,t_dat,price,product_group_name,age_cut,transaction_dat,prev_date,gap,API ratio,total_churn,신규고객이탈
31726158,ffffd9ac14e89946416d80e791d064701994755c3ab686...,0.0,PRE-CREATE,NONE,65.0,2019-12-04,0.084729,Shoes,50대,2019-12-04,0,0.0,0.0,1,1


In [30]:
API_churn_label_merge.columns

Index(['customer_id', 'FN', 'club_member_status', 'fashion_news_frequency',
       'age', 't_dat', 'price', 'product_group_name', 'age_cut',
       'transaction_dat', 'prev_date', 'gap', 'API ratio', 'total_churn',
       '신규고객이탈'],
      dtype='str')

In [31]:
col_kr_map = {
    'customer_id': '고객ID',
    'churn': '이탈여부',  # 0: 잔존, 1: 이탈
    'FN': '패션뉴스구독 여부',  # 패션뉴스구독 여부
    'club_member_status': '멤버십상태',  # ACTIVE, PRE-CREATE, LEFT CLUB
    'fashion_news_frequency': '뉴스레터수신빈도',  # NONE, Monthly, Regularly
    'age': '연령',
    't_dat': '구매일',  # 거래/주문 날짜
    'price': '구매금액',
    'product_group_name': '상품그룹',
    'age_cut': '연령대',
    'prev_date': '이전구매일',  # 이전 구매 날짜
    'gap': '구매간격(일)',  # 현재-이전 구매일 차이
    'total_churn': '전체고객이탈여부'  # 전체고객 기준 이탈/미이탈

}

# 컬럼명 변경
API_churn_label_merge.rename(columns=col_kr_map, inplace=True)

# 확인
API_churn_label_merge.head(50)

,고객ID,패션뉴스구독 여부,멤버십상태,뉴스레터수신빈도,연령,구매일,구매금액,상품그룹,연령대,transaction_dat,이전구매일,구매간격(일),API ratio,전체고객이탈여부,신규고객이탈
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.044051,Garment Upper body,30대,2018-12-27,0,0.0,68.666667,0,0
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.035576,Garment Upper body,30대,2018-12-27,2018-12-27 00:00:00,0.0,68.666667,0,0
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.030492,Garment Upper body,30대,2018-12-27,2018-12-27 00:00:00,0.0,68.666667,0,0
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-02,0.010153,Garment Full body,30대,2019-05-02,2018-12-27 00:00:00,126.0,68.666667,0,0
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-25,0.050831,Garment Upper body,30대,2019-05-25,2019-05-02 00:00:00,23.0,68.666667,0,0
5,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-25,0.050831,Garment Upper body,30대,2019-05-25,2019-05-25 00:00:00,0.0,68.666667,0,0
6,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-07-25,0.012695,Garment Upper body,30대,2019-07-25,2019-05-25 00:00:00,61.0,68.666667,0,0
7,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-07-25,0.021169,Garment Lower body,30대,2019-07-25,2019-07-25 00:00:00,0.0,68.666667,0,0
8,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-09-18,0.044051,Garment Lower body,30대,2019-09-18,2019-07-25 00:00:00,55.0,68.666667,0,0
9,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-09-28,0.054220,Garment Upper body,30대,2019-09-28,2019-09-18 00:00:00,10.0,68.666667,0,0


In [32]:
API_churn_label_merge.to_csv('../../data/processed/final_EDA_df.csv',index=False)